# Task 4: Conditional GAN Optuna Search & Full Retraining

Hyperparameter optimization and full-schedule retraining for the style-conditioned face-to-sketch conditional GAN as specified in the assignment.

### Mathematical Formulation
$$\hat{y} = G(x, s), \quad D(x, s, y), \quad D\big(x, s, G(x,s)\big)$$
$$\mathcal{L}_G = \mathcal{L}_{adv} + \lambda_{L1}\,\mathcal{L}_{L1}\big(y, G(x,s)\big)$$

### Optuna Search Space Requirements
As required by the assignment, Optuna tunes:
1. **Generator Learning Rate** ($G\_lr \in [10^{-5}, 10^{-3}]$ log scale)
2. **Discriminator Learning Rate** ($D\_lr \in [10^{-5}, 10^{-3}]$ log scale)
3. **Batch Size** ($B \in \{4, 8, 16\}$)
4. **Base Channel Count** ($\text{base\_channels} \in \{32, 64\}$)
5. **Dropout Rate** ($\text{dropout} \in [0.0, 0.5]$)
6. **Style-Embedding Dimension** ($\text{embed\_dim} \in \{4, 8, 16\}$)
7. **Reconstruction-Loss Weight** ($\lambda_{L1} \in [10.0, 200.0]$)

### Retraining:
Because GAN search uses reduced epochs (15 epochs), the selected winning configuration is retrained on the complete training schedule (40 epochs), saving `task4_generator_final_best.pt`.

In [ ]:
# 1. Config
N_TRIALS = 12
OPTUNA_EPOCHS = 15
STUDY_NAME = "task4_conditional_gan"
STORAGE_DB = "sqlite:///task4_study.db"
SEED = 42

In [ ]:
# 2. Imports & Setup
import sys
from pathlib import Path
import yaml
import torch
from torch.utils.data import DataLoader
import optuna

RESEARCH_ROOT = Path("..").resolve()
if str(RESEARCH_ROOT) not in sys.path:
    sys.path.insert(0, str(RESEARCH_ROOT))

from constants import CONFIGS_ROOT
from src.device_utils import get_device, device_report
from src.datasets import FS2KDataset
from src.models_gan import UNetGenerator, PatchGANDiscriminator
from src.train_loop import train_one_epoch_gan, validate_gan, set_seed

set_seed(SEED)
device = get_device()

In [ ]:
# 3. Objective Function
train_dataset = FS2KDataset(mode="train", seed=SEED)
val_dataset = FS2KDataset(mode="val", seed=SEED)

def objective(trial: optuna.Trial) -> float:
    g_lr = trial.suggest_float("g_lr", 1e-5, 1e-3, log=True)
    d_lr = trial.suggest_float("d_lr", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [4, 8, 16])
    base_channels = trial.suggest_categorical("base_channels", [32, 64])
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    embed_dim = trial.suggest_categorical("embed_dim", [4, 8, 16])
    lambda_l1 = trial.suggest_float("lambda_l1", 10.0, 200.0)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    G = UNetGenerator(base_channels=base_channels, embed_dim=embed_dim, dropout=dropout).to(device)
    D = PatchGANDiscriminator(base_channels=base_channels).to(device)
    optG = torch.optim.Adam(G.parameters(), lr=g_lr, betas=(0.5, 0.999))
    optD = torch.optim.Adam(D.parameters(), lr=d_lr, betas=(0.5, 0.999))
    best_val_l1 = float("inf")
    for epoch in range(1, OPTUNA_EPOCHS + 1):
        train_metrics = train_one_epoch_gan(G, D, train_loader, optG, optD, device, lambda_l1=lambda_l1)
        vl = validate_gan(G, val_loader, device)
        trial.report(vl["val_l1"], step=epoch)
        trial.set_user_attr(f"epoch_{epoch}_d_real", train_metrics["d_real"])
        trial.set_user_attr(f"epoch_{epoch}_d_fake", train_metrics["d_fake"])
        trial.set_user_attr(f"epoch_{epoch}_g_adv", train_metrics["g_adv"])
        trial.set_user_attr(f"epoch_{epoch}_g_l1", train_metrics["g_l1"])
        best_val_l1 = min(best_val_l1, vl["val_l1"])
        if trial.should_prune():
            raise optuna.TrialPruned()
    return best_val_l1


In [ ]:
# 5. Retrain Winning Conditional GAN on Full Schedule
FULL_TRAIN_EPOCHS = 40
p = study.best_params
print(f"\n--- Retraining Winning Conditional GAN ({FULL_TRAIN_EPOCHS} Epochs) ---")
print(p)

train_loader = DataLoader(train_dataset, batch_size=p["batch_size"], shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=p["batch_size"], shuffle=False, num_workers=2)

winner_G = UNetGenerator(base_channels=p["base_channels"], embed_dim=p["embed_dim"], dropout=p["dropout"]).to(device)
winner_D = PatchGANDiscriminator(base_channels=p["base_channels"]).to(device)
optG = torch.optim.Adam(winner_G.parameters(), lr=p["g_lr"], betas=(0.5, 0.999))
optD = torch.optim.Adam(winner_D.parameters(), lr=p["d_lr"], betas=(0.5, 0.999))

best_l1 = float("inf")
checkpoint_dir = RESEARCH_ROOT / "checkpoints"

for epoch in range(1, FULL_TRAIN_EPOCHS + 1):
    metrics = train_one_epoch_gan(winner_G, winner_D, train_loader, optG, optD, device, lambda_l1=p["lambda_l1"])
    val_metrics = validate_gan(winner_G, val_loader, device)
    is_best = val_metrics["val_l1"] < best_l1
    if is_best:
        best_l1 = val_metrics["val_l1"]
    ckptG = create_checkpoint_dict(
        winner_G, optG, epoch, val_metrics["val_l1"], random_seed=SEED,
        optuna_trial_number=int(study.best_trial.number),
        extra_metadata={**p, "val_l1": val_metrics["val_l1"]}
    )
    save_checkpoint(
        checkpoint_dir, "task4_generator_final", ckptG, is_best=is_best,
        task_name="task4_gan", phase_name="final",
        device_name=device_report()["device_name"]
    )
    print(f"Epoch {epoch:02d}/{FULL_TRAIN_EPOCHS:02d} D-real={metrics['d_real']:.4f} D-fake={metrics['d_fake']:.4f} G-adv={metrics['g_adv']:.4f} G-L1={metrics['g_l1']:.4f} Val-L1={val_metrics['val_l1']:.4f} {'*Best*' if is_best else ''}")

print(f"Final Retrained Generator Best Val L1: {best_l1:.4f}")